In [26]:
import pandas as pd
import numpy as np
import keras
import tensorflow
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Embedding, Flatten, Dense

In [28]:
data = pd.read_csv("train.txt", sep=';')

data.columns = ["Text", "Emotions"]

print(data.head())

                                                Text Emotions
0  i can go from feeling so hopeless to so damned...  sadness
1   im grabbing a minute to post i feel greedy wrong    anger
2  i am ever feeling nostalgic about the fireplac...     love
3                               i am feeling grouchy    anger
4  ive been feeling a little burdened lately wasn...  sadness


In [30]:

texts = data["Text"].tolist()
labels = data["Emotions"].tolist()
# Tokenize the text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)


In [31]:
# Now we need to pad the sequences to the same length to feed them into a neural network. Here’s how we can pad the sequences of the texts to have the same length:
sequences = tokenizer.texts_to_sequences(texts)
max_length = max([len(seq) for seq in sequences])
padded_sequences = pad_sequences(sequences, maxlen=max_length)

In [32]:
# Now I’ll use the label encoder method to convert the classes from strings to a numerical representation:
# Encode the string labels to integers
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)



In [34]:
# We are now going to One-hot encode the labels. One hot encoding refers to the transformation of categorical 
#labels into a binary representation where each label is represented as a vector of all zeros except a single 1. 
#This is necessary because machine learning algorithms work with numerical data. So here is how we can One-hot encode the labels:

# One-hot encode the labels
one_hot_labels = keras.utils.to_categorical(labels)



In [35]:
# Text Emotions Classification Model
# Now we will split the data into training and test sets:
# Split the data into training and testing sets
xtrain, xtest, ytrain, ytest = train_test_split(padded_sequences, 
                                                one_hot_labels, 
                                                test_size=0.2)

In [40]:

# Define the model
model = Sequential()
model.add(Embedding(input_dim=len(tokenizer.word_index) + 1, 
                    output_dim=128, input_length=max_length))
model.add(Flatten())
model.add(Dense(units=128, activation="relu"))
model.add(Dense(units=len(one_hot_labels[0]), activation="softmax"))
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(xtrain, ytrain, epochs=15, batch_size=128, validation_data=(xtest, ytest))


Epoch 1/15


C:\Users\Asus\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.3401 - loss: 1.5815 - val_accuracy: 0.5238 - val_loss: 1.3808
Epoch 2/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - accuracy: 0.6254 - loss: 1.0830 - val_accuracy: 0.7422 - val_loss: 0.7497
Epoch 3/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.9271 - loss: 0.2912 - val_accuracy: 0.7972 - val_loss: 0.5685
Epoch 4/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9822 - loss: 0.0890 - val_accuracy: 0.8050 - val_loss: 0.5723
Epoch 5/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9948 - loss: 0.0368 - val_accuracy: 0.8094 - val_loss: 0.5797
Epoch 6/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9967 - loss: 0.0223 - val_accuracy: 0.8072 - val_loss: 0.6000
Epoch 7/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.9967 - loss: 0.0177 - val_accuracy: 0.8078 - val_loss: 0.6431
Epoch 8/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.9974 - loss: 0.0135 - val_accuracy: 0.804

In [41]:
# Now let’s take a sentence as an input text and see how the model performs:
input_text = "He is admit in hospital today, so he wouldn't be able to show his presence today in the meeting."

In [42]:
# Preprocess the input text
input_sequence = tokenizer.texts_to_sequences([input_text])
padded_input_sequence = pad_sequences(input_sequence, maxlen=max_length)
prediction = model.predict(padded_input_sequence)
predicted_label = label_encoder.inverse_transform([np.argmax(prediction[0])])
print(predicted_label)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
['fear']
